# Cholec80 recurrent temporal benchmark

**Models:** GRU, BiGRU, and Multi-Scale CNN-BiGRU (MS-CBiGRU).

This notebook complements the existing MS-TCN/Transformer experiments with a
recurrent family. Every model receives the same cached ResNet50 features and is
trained with the same split, loss, optimizer, and validation protocol.

## Verified data provenance

The project contains multiple feature lineages:

1. Christian's initial features were generated directly from the original videos
   before the teammates' preprocessing was complete.
2. Giorgia and Marta later produced 1 FPS, 224x224 JPEG frames with aligned phase
   labels (184,578 frames). Christian subsequently extracted another feature set
   from those frames with ImageNet ResNet50.
3. Christian also fine-tuned ResNet50 on videos 1-27 (validation 28-32) and
   extracted a separate fine-tuned feature set directly from the original videos.

Select exactly one lineage below. The notebook never silently mixes feature sets.
Raw-frame extraction is documented but not repeated: it is already complete,
expensive, and would confound the temporal-model comparison.


In [22]:
import gc
import json
import math
import os
import random
import time
import warnings
import zipfile
from copy import deepcopy
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    jaccard_score,
    precision_score,
    recall_score,
)

warnings.filterwarnings("ignore")

# --------------------------- experiment configuration ---------------------------
SEED = 42
FEATURE_SOURCE = "original"  # "teammates", "finetuned", or "original"
MODELS_TO_RUN = ["GRU", "BiGRU", "MS-CBiGRU"]

TRAIN_IDS = list(range(1, 33))
VAL_IDS = list(range(33, 41))
TEST_IDS = list(range(41, 81))
NUM_CLASSES, FEATURE_DIM, FPS = 7, 2048, 1
PHASE_NAMES = [
    "Preparation", "CalotTriangleDissection", "ClippingCutting",
    "GallbladderDissection", "GallbladderPackaging",
    "CleaningCoagulation", "GallbladderRetraction",
]

MAX_EPOCHS, PATIENCE = 80, 12
LR, WEIGHT_DECAY = 3e-4, 1e-5
LABEL_SMOOTHING, TEMPORAL_SMOOTHING_WEIGHT = 0.1, 0.15
PROJECTION_DIM, DROPOUT = 256, 0.30
NORMALIZE_FEATURES = False  # projection LayerNorm is used; enable only as an ablation
USE_AMP = True

RESULTS_DIR = Path(
    f"/kaggle/working/recurrent_benchmark_{FEATURE_SOURCE}"
)
CHECKPOINT_DIR = RESULTS_DIR / "checkpoints"
PREDICTION_DIR = RESULTS_DIR / "predictions"
for directory in (RESULTS_DIR, CHECKPOINT_DIR, PREDICTION_DIR):
    directory.mkdir(parents=True, exist_ok=True)

FEATURE_SOURCES = {
    "teammates": {
        "description": "Giorgia/Marta 1 FPS frames -> ImageNet ResNet50 features",
        "paths": [
            "/kaggle/input/datasets/christiancorvino/features-teammates/features",
            "/kaggle/input/features-teammates/features",
        ],
    },
    "finetuned": {
        "description": "Christian fine-tuned ResNet50 features extracted from original videos",
        "paths": [
            "/kaggle/input/datasets/christiancorvino/features-resnet50finetuned/features_finetuned",
            "/kaggle/input/features-resnet50finetuned/features_finetuned",
            "/kaggle/input/notebooks/christiancorvino/cholec80-resnet50-finetuning/features_finetuned",
        ],
    },
    "original": {
        "description": "Christian initial direct-from-video ImageNet ResNet50 features",
        "paths": [
            "/kaggle/input/datasets/christiancorvino/cholec80-features/features",
            "/kaggle/input/cholec80-features/features",
        ],
    },
}

if FEATURE_SOURCE not in FEATURE_SOURCES:
    raise ValueError(f"Unknown FEATURE_SOURCE={FEATURE_SOURCE!r}")

FEATURE_DIR = next(
    (Path(p) for p in FEATURE_SOURCES[FEATURE_SOURCE]["paths"] if Path(p).is_dir()),
    None,
)
if FEATURE_DIR is None:
    expected = "\n".join(FEATURE_SOURCES[FEATURE_SOURCE]["paths"])
    raise FileNotFoundError(
        f"Attach the Kaggle dataset for FEATURE_SOURCE={FEATURE_SOURCE!r}. Expected one of:\n{expected}"
    )

if torch.cuda.is_available() and torch.cuda.get_device_capability(0) < (7, 0):
    gpu = torch.cuda.get_device_name(0)
    capability = ".".join(map(str, torch.cuda.get_device_capability(0)))
    raise RuntimeError(
        f"Incompatible Kaggle GPU: {gpu} has CUDA capability {capability}. "
        "Select T4 x2 (or another GPU with capability >= 7.0), restart, and Run All."
    )
if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required for the full benchmark. Enable a Kaggle T4 accelerator.")

DEVICE = torch.device("cuda")
USE_AMP = bool(USE_AMP and DEVICE.type == "cuda")
torch.set_float32_matmul_precision("high")

def reset_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

reset_seed()
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA capability:", torch.cuda.get_device_capability(0))
print("PyTorch / CUDA:", torch.__version__, "/", torch.version.cuda)
print("Feature source:", FEATURE_SOURCE)
print("Lineage:", FEATURE_SOURCES[FEATURE_SOURCE]["description"])
print("Directory:", FEATURE_DIR)
print("Split: 32 train / 8 validation / 40 test")


GPU: Tesla T4
CUDA capability: (7, 5)
PyTorch / CUDA: 2.10.0+cu128 / 12.8
Feature source: original
Lineage: Christian initial direct-from-video ImageNet ResNet50 features
Directory: /kaggle/input/datasets/christiancorvino/cholec80-features/features
Split: 32 train / 8 validation / 40 test


## 1. Executed preprocessing: feature integrity and train-only statistics

The expensive visual preprocessing has already been performed upstream:

```text
original videos + phase annotations
      -> sample at 1 FPS
      -> resize to 224x224
      -> ResNet50
      -> one 2048-D vector and one phase label per second
```

This notebook executes the temporal preprocessing below. Test features are not
loaded until the final evaluation cell.


In [23]:
def feature_paths(video_id):
    return (
        FEATURE_DIR / f"video{video_id:02d}_features.npy",
        FEATURE_DIR / f"video{video_id:02d}_labels.npy",
    )

inventory = []
for video_id in range(1, 81):
    feature_path, label_path = feature_paths(video_id)
    if not feature_path.exists() or not label_path.exists():
        raise FileNotFoundError(f"Missing feature/label pair for video {video_id:02d}")
    features = np.load(feature_path, mmap_mode="r")
    labels = np.load(label_path, mmap_mode="r")
    if features.ndim != 2 or features.shape[1] != FEATURE_DIM:
        raise ValueError(f"video{video_id:02d}: invalid feature shape {features.shape}")
    if labels.ndim != 1 or len(features) != len(labels):
        raise ValueError(f"video{video_id:02d}: shapes do not align: {features.shape}, {labels.shape}")
    if not np.isfinite(features).all():
        raise ValueError(f"video{video_id:02d}: NaN or infinity in features")
    if labels.min() < 0 or labels.max() >= NUM_CLASSES:
        raise ValueError(f"video{video_id:02d}: invalid phase IDs")
    inventory.append({
        "video": video_id,
        "samples": len(labels),
        "minutes": len(labels) / (60 * FPS),
        "feature_dim": features.shape[1],
        "phases_present": len(np.unique(labels)),
    })

inventory = pd.DataFrame(inventory)
display(inventory)
display(inventory.describe())
inventory.to_csv(RESULTS_DIR / "feature_inventory.csv", index=False)

def calculate_train_stats(video_ids):
    total = 0
    sum_x = np.zeros(FEATURE_DIM, dtype=np.float64)
    sum_x2 = np.zeros(FEATURE_DIM, dtype=np.float64)
    for video_id in video_ids:
        x = np.asarray(np.load(feature_paths(video_id)[0], mmap_mode="r"), dtype=np.float64)
        total += len(x)
        sum_x += x.sum(axis=0)
        sum_x2 += np.square(x).sum(axis=0)
    mean = sum_x / total
    std = np.sqrt(np.maximum(sum_x2 / total - np.square(mean), 1e-8))
    return mean.astype(np.float32), std.astype(np.float32)

FEATURE_MEAN, FEATURE_STD = (
    calculate_train_stats(TRAIN_IDS) if NORMALIZE_FEATURES else (None, None)
)

class FeatureSequenceDataset:
    def __init__(self, video_ids):
        self.video_ids = list(video_ids)
        self.data = []
        for video_id in self.video_ids:
            x = np.load(feature_paths(video_id)[0]).astype(np.float32)
            y = np.load(feature_paths(video_id)[1]).astype(np.int64)
            if NORMALIZE_FEATURES:
                x = (x - FEATURE_MEAN) / FEATURE_STD
            self.data.append({
                "video_id": video_id,
                "features": torch.from_numpy(x),
                "labels": torch.from_numpy(y),
            })

    def __len__(self):
        return len(self.data)

train_ds = FeatureSequenceDataset(TRAIN_IDS)
val_ds = FeatureSequenceDataset(VAL_IDS)

train_labels = np.concatenate([item["labels"].numpy() for item in train_ds.data])
class_counts = np.bincount(train_labels, minlength=NUM_CLASSES)
class_weights = len(train_labels) / (NUM_CLASSES * np.maximum(class_counts, 1))
class_weights = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
print("Training samples:", len(train_labels))
print("Class counts:", class_counts.tolist())
print("Class weights:", class_weights.cpu().numpy().round(3).tolist())


,video,samples,minutes,feature_dim,phases_present
0,1,1734,28.900000,2048,7
1,2,2840,47.333333,2048,7
2,3,5829,97.150000,2048,7
3,4,1523,25.383333,2048,7
4,5,2345,39.083333,2048,7
...,...,...,...,...,...
75,76,2650,44.166667,2048,7
76,77,2503,41.716667,2048,7
77,78,740,12.333333,2048,7
78,79,3415,56.916667,2048,7


,video,samples,minutes,feature_dim,phases_present
count,80.0000,80.000000,80.000000,80.0,80.000000
mean,40.5000,2307.225000,38.453750,2048.0,6.812500
std,23.2379,1025.529057,17.092151,0.0,0.392775
min,1.0000,740.000000,12.333333,2048.0,6.000000
25%,20.7500,1645.000000,27.416667,2048.0,7.000000
50%,40.5000,2096.000000,34.933333,2048.0,7.000000
75%,60.2500,2861.500000,47.691667,2048.0,7.000000
max,80.0000,5994.000000,99.900000,2048.0,7.000000


Training samples: 71032
Class counts: [3104, 29730, 6139, 20046, 3084, 5979, 2950]
Class weights: [3.2690000534057617, 0.3409999907016754, 1.652999997138977, 0.5059999823570251, 3.2899999618530273, 1.6970000267028809, 3.440000057220459]


## 2. Framewise and temporal metrics

Reported metrics:

- Accuracy, macro precision, macro recall, macro F1, macro Jaccard
- Mean segmental Edit score
- Segmental F1@10, F1@25, and F1@50

All metrics are stored on the [0, 1] scale, matching Christian's notebooks.

Segment extraction is performed independently for every video. Video boundaries
are never concatenated into a single artificial sequence.


In [24]:
def extract_segments(sequence):
    sequence = np.asarray(sequence, dtype=np.int64)
    if len(sequence) == 0:
        return np.array([], int), np.array([], int), np.array([], int)
    change = np.r_[True, sequence[1:] != sequence[:-1]]
    starts = np.flatnonzero(change)
    ends = np.r_[starts[1:], len(sequence)]
    return sequence[starts], starts, ends

def normalized_edit_score(target, prediction):
    target_segments = extract_segments(target)[0].tolist()
    pred_segments = extract_segments(prediction)[0].tolist()
    previous = list(range(len(pred_segments) + 1))
    for i, target_label in enumerate(target_segments, 1):
        current = [i]
        for j, pred_label in enumerate(pred_segments, 1):
            current.append(min(
                current[-1] + 1,
                previous[j] + 1,
                previous[j - 1] + int(target_label != pred_label),
            ))
        previous = current
    distance = previous[-1]
    return 1.0 - distance / max(len(target_segments), len(pred_segments), 1)

def segmental_f1_counts(target, prediction, overlap):
    gt_labels, gt_starts, gt_ends = extract_segments(target)
    pr_labels, pr_starts, pr_ends = extract_segments(prediction)
    matched = np.zeros(len(gt_labels), dtype=bool)
    tp = fp = 0
    for label, start, end in zip(pr_labels, pr_starts, pr_ends):
        intersection = np.minimum(end, gt_ends) - np.maximum(start, gt_starts)
        union = np.maximum(end, gt_ends) - np.minimum(start, gt_starts)
        iou = (intersection / np.maximum(union, 1)) * (gt_labels == label)
        best = int(np.argmax(iou)) if len(iou) else -1
        if best >= 0 and iou[best] >= overlap and not matched[best]:
            tp += 1
            matched[best] = True
        else:
            fp += 1
    fn = int((~matched).sum())
    return tp, fp, fn

def evaluate_sequences(results, prediction_key="preds"):
    all_target, all_prediction, edits = [], [], []
    counts = {0.10: [0, 0, 0], 0.25: [0, 0, 0], 0.50: [0, 0, 0]}
    for video_id in sorted(results):
        target = np.asarray(results[video_id]["labels"])
        prediction = np.asarray(results[video_id][prediction_key])
        all_target.append(target)
        all_prediction.append(prediction)
        edits.append(normalized_edit_score(target, prediction))
        for overlap in counts:
            values = segmental_f1_counts(target, prediction, overlap)
            counts[overlap] = [a + b for a, b in zip(counts[overlap], values)]

    target = np.concatenate(all_target)
    prediction = np.concatenate(all_prediction)
    metrics = {
        "accuracy": accuracy_score(target, prediction),
        "precision_macro": precision_score(
            target, prediction, labels=np.arange(NUM_CLASSES), average="macro", zero_division=0),
        "recall_macro": recall_score(
            target, prediction, labels=np.arange(NUM_CLASSES), average="macro", zero_division=0),
        "f1_macro": f1_score(
            target, prediction, labels=np.arange(NUM_CLASSES), average="macro", zero_division=0),
        "jaccard_macro": jaccard_score(
            target, prediction, labels=np.arange(NUM_CLASSES), average="macro", zero_division=0),
        "edit": float(np.mean(edits)),
    }
    for overlap, (tp, fp, fn) in counts.items():
        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1)
        metrics[f"f1_at_{int(overlap * 100)}"] = (
            2 * precision * recall / max(precision + recall, 1e-12)
        )
    metrics["f1_segmental_avg"] = np.mean([
        metrics["f1_at_10"], metrics["f1_at_25"], metrics["f1_at_50"]
    ])
    return metrics

def per_video_metrics(results, prediction_key="preds"):
    rows = []
    for video_id in sorted(results):
        one = {video_id: results[video_id]}
        metrics = evaluate_sequences(one, prediction_key)
        rows.append({"video": video_id, **metrics})
    return pd.DataFrame(rows)


## 3. Three complementary recurrent models

- **GRU:** causal recurrent memory.
- **BiGRU:** past and future context for offline recognition.
- **MS-CBiGRU:** multi-scale dilated convolutions before the BiGRU, testing
  whether local transition patterns complement recurrent workflow memory.


In [25]:
class InputProjection(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(FEATURE_DIM, PROJECTION_DIM),
            nn.LayerNorm(PROJECTION_DIM),
            nn.GELU(),
            nn.Dropout(DROPOUT),
        )

    def forward(self, x):
        return self.net(x)

class GRUPhaseModel(nn.Module):
    def __init__(self, bidirectional=False):
        super().__init__()
        self.project = InputProjection()
        hidden = PROJECTION_DIM // 2 if bidirectional else PROJECTION_DIM
        self.gru = nn.GRU(
            input_size=PROJECTION_DIM,
            hidden_size=hidden,
            num_layers=2,
            batch_first=True,
            dropout=DROPOUT,
            bidirectional=bidirectional,
        )
        output_dim = hidden * (2 if bidirectional else 1)
        self.classifier = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(output_dim, NUM_CLASSES))

    def forward(self, x):
        x = self.project(x)
        x, _ = self.gru(x)
        return self.classifier(x)

class MultiScaleTemporalBlock(nn.Module):
    def __init__(self):
        super().__init__()
        branch_dim = PROJECTION_DIM // 4
        self.branches = nn.ModuleList([
            nn.Conv1d(PROJECTION_DIM, branch_dim, kernel_size=3,
                      padding=dilation, dilation=dilation)
            for dilation in (1, 2, 4, 8)
        ])
        self.mix = nn.Sequential(
            nn.Conv1d(PROJECTION_DIM, PROJECTION_DIM, kernel_size=1),
            nn.GELU(),
            nn.Dropout(DROPOUT),
        )
        self.norm = nn.LayerNorm(PROJECTION_DIM)

    def forward(self, x):
        residual = x
        x = x.transpose(1, 2)
        x = torch.cat([F.gelu(branch(x)) for branch in self.branches], dim=1)
        x = self.mix(x).transpose(1, 2)
        return self.norm(x + residual)

class MSCBiGRU(nn.Module):
    def __init__(self):
        super().__init__()
        self.project = InputProjection()
        self.temporal = MultiScaleTemporalBlock()
        self.gru = nn.GRU(
            input_size=PROJECTION_DIM,
            hidden_size=PROJECTION_DIM // 2,
            num_layers=2,
            batch_first=True,
            dropout=DROPOUT,
            bidirectional=True,
        )
        self.classifier = nn.Sequential(
            nn.Dropout(DROPOUT), nn.Linear(PROJECTION_DIM, NUM_CLASSES)
        )

    def forward(self, x):
        x = self.temporal(self.project(x))
        x, _ = self.gru(x)
        return self.classifier(x)

MODEL_BUILDERS = {
    "GRU": lambda: GRUPhaseModel(bidirectional=False),
    "BiGRU": lambda: GRUPhaseModel(bidirectional=True),
    "MS-CBiGRU": MSCBiGRU,
}

for name in MODELS_TO_RUN:
    if name not in MODEL_BUILDERS:
        raise ValueError(f"Unknown model {name!r}")
    model = MODEL_BUILDERS[name]()
    parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"{name:12s}: {parameters:,} trainable parameters")
    del model


GRU         : 1,316,359 trainable parameters
BiGRU       : 1,119,751 trainable parameters
MS-CBiGRU   : 1,382,919 trainable parameters


## 4. Training and validation-only model/post-processing selection

The best checkpoint is selected by validation average segmental F1. The transition
matrix is learned from training labels only. Raw versus Viterbi output is selected
independently for each model on validation; test remains untouched.


In [26]:
class TemporalSmoothingLoss(nn.Module):
    def forward(self, logits):
        log_probabilities = F.log_softmax(logits, dim=-1)
        differences = log_probabilities[:, 1:] - log_probabilities[:, :-1]
        return torch.clamp(differences.square(), max=16.0).mean()

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
smoothing_loss = TemporalSmoothingLoss()

def combined_loss(logits, labels):
    classification = criterion(logits.transpose(1, 2), labels)
    temporal = smoothing_loss(logits)
    return classification + TEMPORAL_SMOOTHING_WEIGHT * temporal

@torch.no_grad()
def predict_dataset(model, dataset):
    model.eval()
    results = {}
    start = time.perf_counter()
    for item in dataset.data:
        x = item["features"].unsqueeze(0).to(DEVICE, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            logits = model(x).squeeze(0)
        log_probabilities = F.log_softmax(logits.float(), dim=-1).cpu().numpy()
        results[item["video_id"]] = {
            "labels": item["labels"].numpy(),
            "preds": log_probabilities.argmax(axis=1),
            "log_probs": log_probabilities,
        }
    return results, time.perf_counter() - start

@torch.no_grad()
def validation_loss(model, dataset):
    model.eval()
    values = []
    for item in dataset.data:
        x = item["features"].unsqueeze(0).to(DEVICE, non_blocking=True)
        y = item["labels"].unsqueeze(0).to(DEVICE, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            values.append(combined_loss(model(x), y).item())
    return float(np.mean(values))

def train_one_model(model_name):
    reset_seed()
    model = MODEL_BUILDERS[model_name]().to(DEVICE)
    parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=5, min_lr=1e-6
    )
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    history, best_state = [], None
    best_primary, stale = -np.inf, 0
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    training_start = time.perf_counter()

    print(f"\n===== Training {model_name} =====")
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        losses = []
        for index in np.random.permutation(len(train_ds.data)):
            item = train_ds.data[index]
            x = item["features"].unsqueeze(0).to(DEVICE, non_blocking=True)
            y = item["labels"].unsqueeze(0).to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                logits = model(x)
                loss = combined_loss(logits, y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(optimizer)
            scaler.update()
            losses.append(loss.item())

        val_results, _ = predict_dataset(model, val_ds)
        val_metrics = evaluate_sequences(val_results)
        val_loss = validation_loss(model, val_ds)
        scheduler.step(val_loss)
        primary = val_metrics["f1_segmental_avg"]
        history.append({
            "epoch": epoch,
            "train_loss": float(np.mean(losses)),
            "val_loss": val_loss,
            "val_f1_segmental_avg": primary,
            "val_edit": val_metrics["edit"],
            "val_f1_macro": val_metrics["f1_macro"],
            "lr": optimizer.param_groups[0]["lr"],
        })

        if primary > best_primary:
            best_primary = primary
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            stale = 0
            torch.save(best_state, CHECKPOINT_DIR / f"best_{model_name.lower().replace('-', '_')}.pt")
        else:
            stale += 1

        print(
            f"{model_name:11s} epoch={epoch:03d} train={np.mean(losses):.4f} "
            f"val={val_loss:.4f} F1avg={primary:.2f} Edit={val_metrics['edit']:.2f}"
        )
        if stale >= PATIENCE:
            print(f"Early stopping after {epoch} epochs")
            break

    model.load_state_dict(best_state)
    training_seconds = time.perf_counter() - training_start
    peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
    return model, pd.DataFrame(history), {
        "parameters": parameter_count,
        "training_seconds": training_seconds,
        "peak_vram_mb": peak_vram_mb,
        "best_val_f1_segmental_avg": best_primary,
    }

def training_transition_model(dataset):
    transitions = np.full((NUM_CLASSES, NUM_CLASSES), 1e-3, dtype=np.float64)
    starts = np.full(NUM_CLASSES, 1e-3, dtype=np.float64)
    for item in dataset.data:
        labels = item["labels"].numpy()
        starts[labels[0]] += 1
        for previous, current in zip(labels[:-1], labels[1:]):
            transitions[previous, current] += 1
    transitions /= transitions.sum(axis=1, keepdims=True)
    starts /= starts.sum()
    return np.log(transitions), np.log(starts)

TRANSITION_LOG, START_LOG = training_transition_model(train_ds)

def viterbi_decode(log_probabilities):
    length, classes = log_probabilities.shape
    score = np.full((length, classes), -np.inf, dtype=np.float64)
    back = np.zeros((length, classes), dtype=np.int64)
    score[0] = START_LOG + log_probabilities[0]
    for time_index in range(1, length):
        candidates = score[time_index - 1][:, None] + TRANSITION_LOG
        back[time_index] = candidates.argmax(axis=0)
        score[time_index] = candidates.max(axis=0) + log_probabilities[time_index]
    path = np.zeros(length, dtype=np.int64)
    path[-1] = score[-1].argmax()
    for time_index in range(length - 2, -1, -1):
        path[time_index] = back[time_index + 1, path[time_index + 1]]
    return path

def add_postprocessing(results):
    output = deepcopy(results)
    for item in output.values():
        item["viterbi"] = viterbi_decode(item["log_probs"])
    return output

TRAINING_SUMMARIES = {}
VALIDATION_SELECTION = {}

for model_name in MODELS_TO_RUN:
    model, history, summary = train_one_model(model_name)
    history.to_csv(RESULTS_DIR / f"training_history_{model_name.lower().replace('-', '_')}.csv", index=False)

    validation_results, validation_inference_seconds = predict_dataset(model, val_ds)
    validation_results = add_postprocessing(validation_results)
    raw_metrics = evaluate_sequences(validation_results, "preds")
    viterbi_metrics = evaluate_sequences(validation_results, "viterbi")
    selection_table = pd.DataFrame([
        {"postprocessing": "raw", **raw_metrics},
        {"postprocessing": "viterbi", **viterbi_metrics},
    ]).sort_values(["f1_segmental_avg", "edit"], ascending=False).reset_index(drop=True)
    selected = selection_table.iloc[0]["postprocessing"]
    selection_table.to_csv(
        RESULTS_DIR / f"validation_postprocessing_{model_name.lower().replace('-', '_')}.csv",
        index=False,
    )
    display(selection_table)
    print(f"{model_name}: selected on validation -> {selected}")

    model_key = model_name.lower().replace("-", "_")
    for video_id, values in validation_results.items():
        np.savez_compressed(
            PREDICTION_DIR / f"{model_key}_val_video{video_id:02d}.npz",
            labels=values["labels"],
            log_probs=values["log_probs"],
            raw=values["preds"],
            viterbi=values["viterbi"],
        )

    TRAINING_SUMMARIES[model_name] = {
        **summary,
        "validation_inference_seconds": validation_inference_seconds,
    }
    VALIDATION_SELECTION[model_name] = selected
    del model
    gc.collect()
    torch.cuda.empty_cache()

print("Frozen validation decisions:", VALIDATION_SELECTION)



===== Training GRU =====
GRU         epoch=001 train=1.9261 val=1.8091 F1avg=0.05 Edit=0.06
GRU         epoch=002 train=1.6595 val=1.7032 F1avg=0.05 Edit=0.06
GRU         epoch=003 train=1.5208 val=1.6420 F1avg=0.09 Edit=0.09
GRU         epoch=004 train=1.4336 val=1.4565 F1avg=0.09 Edit=0.08
GRU         epoch=005 train=1.2759 val=1.3438 F1avg=0.16 Edit=0.13
GRU         epoch=006 train=1.1826 val=1.2664 F1avg=0.21 Edit=0.16
GRU         epoch=007 train=1.0853 val=1.3238 F1avg=0.25 Edit=0.26
GRU         epoch=008 train=1.0455 val=1.2734 F1avg=0.18 Edit=0.18
GRU         epoch=009 train=0.9776 val=1.2700 F1avg=0.30 Edit=0.31
GRU         epoch=010 train=0.9343 val=1.2851 F1avg=0.32 Edit=0.25
GRU         epoch=011 train=0.9057 val=1.2251 F1avg=0.24 Edit=0.23
GRU         epoch=012 train=0.9428 val=1.2713 F1avg=0.30 Edit=0.25
GRU         epoch=013 train=0.8769 val=1.1733 F1avg=0.32 Edit=0.28
GRU         epoch=014 train=0.8588 val=1.2804 F1avg=0.25 Edit=0.20
GRU         epoch=015 train=0.8580 v

,postprocessing,accuracy,precision_macro,recall_macro,f1_macro,jaccard_macro,edit,f1_at_10,f1_at_25,f1_at_50,f1_segmental_avg
0,viterbi,0.862526,0.856557,0.885298,0.862455,0.761755,0.770662,0.842975,0.826446,0.743802,0.804408
1,raw,0.846787,0.831022,0.869637,0.840884,0.729726,0.398376,0.476190,0.466667,0.390476,0.444444


GRU: selected on validation -> viterbi

===== Training BiGRU =====
BiGRU       epoch=001 train=1.9055 val=1.7735 F1avg=0.05 Edit=0.06
BiGRU       epoch=002 train=1.6201 val=1.6380 F1avg=0.06 Edit=0.07
BiGRU       epoch=003 train=1.4545 val=1.5692 F1avg=0.13 Edit=0.12
BiGRU       epoch=004 train=1.3525 val=1.4658 F1avg=0.17 Edit=0.14
BiGRU       epoch=005 train=1.1934 val=1.2908 F1avg=0.27 Edit=0.22
BiGRU       epoch=006 train=1.0943 val=1.2811 F1avg=0.35 Edit=0.29
BiGRU       epoch=007 train=1.0160 val=1.1506 F1avg=0.54 Edit=0.51
BiGRU       epoch=008 train=0.9633 val=1.1391 F1avg=0.56 Edit=0.50
BiGRU       epoch=009 train=0.9045 val=1.1396 F1avg=0.63 Edit=0.62
BiGRU       epoch=010 train=0.8829 val=1.0794 F1avg=0.68 Edit=0.63
BiGRU       epoch=011 train=0.8619 val=1.1162 F1avg=0.59 Edit=0.48
BiGRU       epoch=012 train=0.8542 val=1.0865 F1avg=0.66 Edit=0.59
BiGRU       epoch=013 train=0.8381 val=1.0306 F1avg=0.64 Edit=0.57
BiGRU       epoch=014 train=0.8206 val=1.0689 F1avg=0.61 Edit=

,postprocessing,accuracy,precision_macro,recall_macro,f1_macro,jaccard_macro,edit,f1_at_10,f1_at_25,f1_at_50,f1_segmental_avg
0,viterbi,0.885449,0.790558,0.926974,0.835469,0.725566,0.894345,0.928571,0.875000,0.785714,0.863095
1,raw,0.874739,0.774187,0.919940,0.822518,0.707094,0.728743,0.818898,0.787402,0.692913,0.766404


BiGRU: selected on validation -> viterbi

===== Training MS-CBiGRU =====
MS-CBiGRU   epoch=001 train=1.8914 val=1.7722 F1avg=0.06 Edit=0.07
MS-CBiGRU   epoch=002 train=1.6087 val=1.6237 F1avg=0.07 Edit=0.08
MS-CBiGRU   epoch=003 train=1.4607 val=1.6413 F1avg=0.15 Edit=0.17
MS-CBiGRU   epoch=004 train=1.3434 val=1.5241 F1avg=0.20 Edit=0.16
MS-CBiGRU   epoch=005 train=1.1910 val=1.2552 F1avg=0.34 Edit=0.30
MS-CBiGRU   epoch=006 train=1.0821 val=1.2382 F1avg=0.44 Edit=0.39
MS-CBiGRU   epoch=007 train=1.0235 val=1.2130 F1avg=0.56 Edit=0.53
MS-CBiGRU   epoch=008 train=0.9398 val=1.1678 F1avg=0.63 Edit=0.60
MS-CBiGRU   epoch=009 train=0.9065 val=1.2438 F1avg=0.62 Edit=0.62
MS-CBiGRU   epoch=010 train=0.9110 val=1.2208 F1avg=0.68 Edit=0.65
MS-CBiGRU   epoch=011 train=0.8697 val=1.1775 F1avg=0.67 Edit=0.62
MS-CBiGRU   epoch=012 train=0.8409 val=1.0453 F1avg=0.67 Edit=0.64
MS-CBiGRU   epoch=013 train=0.8403 val=1.0854 F1avg=0.68 Edit=0.65
MS-CBiGRU   epoch=014 train=0.8276 val=1.1960 F1avg=0.56

,postprocessing,accuracy,precision_macro,recall_macro,f1_macro,jaccard_macro,edit,f1_at_10,f1_at_25,f1_at_50,f1_segmental_avg
0,viterbi,0.918691,0.877235,0.940240,0.901573,0.825094,0.923115,0.945455,0.945455,0.854545,0.915152
1,raw,0.912291,0.864868,0.937981,0.892921,0.811086,0.825019,0.873950,0.857143,0.773109,0.834734


MS-CBiGRU: selected on validation -> viterbi
Frozen validation decisions: {'GRU': 'viterbi', 'BiGRU': 'viterbi', 'MS-CBiGRU': 'viterbi'}


## 5. Final test evaluation

This is the first cell that loads test features. All architectures, checkpoints,
and post-processing choices have already been frozen. Each declared model is
evaluated once, and test results are not used for further tuning.


In [27]:
test_ds = FeatureSequenceDataset(TEST_IDS)
comparison_rows = []
all_per_video = []

for model_name in MODELS_TO_RUN:
    model = MODEL_BUILDERS[model_name]().to(DEVICE)
    model_key = model_name.lower().replace("-", "_")
    state = torch.load(CHECKPOINT_DIR / f"best_{model_key}.pt", map_location=DEVICE)
    model.load_state_dict(state)

    test_results, test_inference_seconds = predict_dataset(model, test_ds)
    test_results = add_postprocessing(test_results)
    selected_key = "preds" if VALIDATION_SELECTION[model_name] == "raw" else "viterbi"
    metrics = evaluate_sequences(test_results, selected_key)
    summary = TRAINING_SUMMARIES[model_name]
    comparison_rows.append({
        "model": model_name,
        "feature_source": FEATURE_SOURCE,
        "postprocessing": VALIDATION_SELECTION[model_name],
        "parameters": summary["parameters"],
        "training_minutes": summary["training_seconds"] / 60,
        "peak_vram_mb": summary["peak_vram_mb"],
        "test_inference_seconds": test_inference_seconds,
        **metrics,
    })

    video_table = per_video_metrics(test_results, selected_key)
    video_table.insert(0, "model", model_name)
    all_per_video.append(video_table)

    for video_id, values in test_results.items():
        np.savez_compressed(
            PREDICTION_DIR / f"{model_key}_test_video{video_id:02d}.npz",
            labels=values["labels"],
            log_probs=values["log_probs"],
            raw=values["preds"],
            viterbi=values["viterbi"],
            selected=values[selected_key],
        )

    target = np.concatenate([test_results[v]["labels"] for v in sorted(test_results)])
    prediction = np.concatenate([test_results[v][selected_key] for v in sorted(test_results)])
    matrix = confusion_matrix(target, prediction, labels=np.arange(NUM_CLASSES), normalize="true")
    fig, ax = plt.subplots(figsize=(9, 8))
    image = ax.imshow(matrix, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(NUM_CLASSES), PHASE_NAMES, rotation=55, ha="right")
    ax.set_yticks(range(NUM_CLASSES), PHASE_NAMES)
    ax.set_xlabel("Predicted phase")
    ax.set_ylabel("True phase")
    ax.set_title(f"{model_name} normalized confusion matrix")
    fig.colorbar(image, ax=ax)
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / f"confusion_matrix_{model_key}.png", dpi=160)
    plt.close(fig)

    del model, test_results
    gc.collect()
    torch.cuda.empty_cache()

comparison = pd.DataFrame(comparison_rows).sort_values(
    ["f1_segmental_avg", "edit"], ascending=False
).reset_index(drop=True)
per_video = pd.concat(all_per_video, ignore_index=True)

display(comparison)
comparison.to_csv(RESULTS_DIR / "final_model_comparison.csv", index=False)
per_video.to_csv(RESULTS_DIR / "test_metrics_per_video.csv", index=False)

protocol = {
    "seed": SEED,
    "feature_source": FEATURE_SOURCE,
    "feature_lineage": FEATURE_SOURCES[FEATURE_SOURCE]["description"],
    "feature_directory": str(FEATURE_DIR),
    "split": {"train": TRAIN_IDS, "validation": VAL_IDS, "test": TEST_IDS},
    "models": MODELS_TO_RUN,
    "validation_selected_postprocessing": VALIDATION_SELECTION,
    "training": {
        "max_epochs": MAX_EPOCHS,
        "patience": PATIENCE,
        "learning_rate": LR,
        "weight_decay": WEIGHT_DECAY,
        "label_smoothing": LABEL_SMOOTHING,
        "temporal_smoothing_weight": TEMPORAL_SMOOTHING_WEIGHT,
        "normalization": NORMALIZE_FEATURES,
        "amp": USE_AMP,
    },
    "selection_rule": "checkpoint and post-processing selected by validation average F1@10/25/50; test once",
}
with open(RESULTS_DIR / "protocol.json", "w") as file:
    json.dump(protocol, file, indent=2)

zip_path = Path(
    f"/kaggle/working/cholec80_recurrent_benchmark_"
    f"{FEATURE_SOURCE}_results.zip"
)
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in RESULTS_DIR.rglob("*"):
        if path.is_file():
            archive.write(path, path.relative_to(RESULTS_DIR.parent))

print("Saved final archive:", zip_path)
print("Primary result:", RESULTS_DIR / "final_model_comparison.csv")


,model,feature_source,postprocessing,parameters,training_minutes,peak_vram_mb,test_inference_seconds,accuracy,precision_macro,recall_macro,f1_macro,jaccard_macro,edit,f1_at_10,f1_at_25,f1_at_50,f1_segmental_avg
0,MS-CBiGRU,original,viterbi,1382919,0.580360,282.637695,0.678289,0.880693,0.811528,0.891149,0.837517,0.728283,0.857075,0.896667,0.876667,0.796667,0.856667
1,BiGRU,original,viterbi,1119751,0.467557,251.740234,0.719824,0.810962,0.732976,0.867676,0.771641,0.637598,0.790870,0.836193,0.792512,0.673947,0.767551
2,GRU,original,viterbi,1316359,2.138959,205.185547,2.279320,0.804355,0.764797,0.834511,0.783079,0.650060,0.650571,0.729885,0.715517,0.626437,0.690613


Saved final archive: /kaggle/working/cholec80_recurrent_benchmark_original_results.zip
Primary result: /kaggle/working/recurrent_benchmark_original/final_model_comparison.csv
